# 🌳 Decision Tree — Capstone Solutions
**Topic:** Complete Answers to All Exercise Questions

---

## 📌 Yeh File Kya Hai?

Notebook 01 aur 02 mein jo Capstone/TODO exercises the, unke **complete solutions** yahan hain.

| Notebook | Parts | Dataset |
|----------|-------|---------|
| 01_Concepts | Heart Disease A, B, C, D | Synthetic heart data |
| 02_Applied | Iris A, B, C, D | sklearn Iris |

---
⚠️ **Pehle khud try karo, phir yahan dekho!**

---
# 📒 Notebook 01 Solutions: Heart Disease Capstone

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Same data recreation from notebook 01
np.random.seed(42)
n = 303
df_heart = pd.DataFrame({
    'age':      np.random.randint(29, 77, n),
    'sex':      np.random.randint(0, 2, n),
    'cp':       np.random.randint(0, 4, n),
    'trestbps': np.random.randint(94, 200, n),
    'chol':     np.random.randint(126, 564, n),
    'fbs':      np.random.randint(0, 2, n),
    'restecg':  np.random.randint(0, 3, n),
    'thalach':  np.random.randint(71, 202, n),
    'exang':    np.random.randint(0, 2, n),
    'oldpeak':  np.round(np.random.uniform(0, 6.2, n), 1),
    'slope':    np.random.randint(0, 3, n),
    'ca':       np.random.randint(0, 4, n),
    'thal':     np.random.choice([0, 1, 2, 3], n),
})
score = (df_heart.age > 55).astype(int) + (df_heart.trestbps > 140).astype(int) + \
        (df_heart.chol > 240).astype(int) + (df_heart.exang == 1).astype(int) + \
        (df_heart.ca > 0).astype(int)
df_heart['target'] = (score >= 2).astype(int)

X_h = df_heart.drop('target', axis=1)
y_h = df_heart['target']
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42, stratify=y_h
)
print(f"Data ready: {df_heart.shape} | Train: {X_train_h.shape[0]} | Test: {X_test_h.shape[0]}")

---
## ✅ Part A — EDA + Data Prep

In [ ]:
# TODO 1: EDA
print("=== Basic Info ===")
print(f"Shape: {df_heart.shape}")
print(f"\nMissing values:\n{df_heart.isnull().sum().to_string()}")
print(f"\nTarget distribution:\n{df_heart.target.value_counts()}")
print(f"\nBasic stats:")
print(df_heart.describe().round(2).T[['mean','std','min','max']].to_string())

In [ ]:
# TODO 2: Distribution plots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Age by target
df_heart[df_heart.target==1]['age'].plot(kind='hist', ax=axes[0,0], alpha=0.6,
                                          label='Disease', color='#e74c3c', bins=15)
df_heart[df_heart.target==0]['age'].plot(kind='hist', ax=axes[0,0], alpha=0.6,
                                          label='No Disease', color='#2ecc71', bins=15)
axes[0,0].set_title('Age Distribution by Target')
axes[0,0].legend()

# Target balance
df_heart['target'].value_counts().plot(kind='pie', ax=axes[0,1],
    labels=['Disease', 'No Disease'], autopct='%1.1f%%', colors=['#e74c3c','#2ecc71'])
axes[0,1].set_title('Target Distribution')
axes[0,1].set_ylabel('')

# Age vs thalach (scatter)
scatter = axes[0,2].scatter(df_heart.age, df_heart.thalach,
                             c=df_heart.target, cmap='RdYlGn', alpha=0.6)
axes[0,2].set_xlabel('Age')
axes[0,2].set_ylabel('Max Heart Rate (thalach)')
axes[0,2].set_title('Age vs Thalach (Green=No Disease)')

# ca (vessels) by target
ca_surv = df_heart.groupby('ca')['target'].mean()
ca_surv.plot(kind='bar', ax=axes[1,0], color='#9b59b6', rot=0)
axes[1,0].set_title('Disease Rate by CA (Vessels)')
axes[1,0].set_xlabel('Number of Major Vessels')

# exang
exang_disease = df_heart.groupby('exang')['target'].mean()
exang_disease.plot(kind='bar', ax=axes[1,1],
                   color=['#2ecc71', '#e74c3c'], rot=0)
axes[1,1].set_title('Disease Rate: Exercise Angina')
axes[1,1].set_xlabel('Exang (0=No, 1=Yes)')

# chol distribution
df_heart.boxplot(column='chol', by='target', ax=axes[1,2])
axes[1,2].set_title('Cholesterol by Target')
axes[1,2].set_xlabel('Target (0=No Disease, 1=Disease)')

plt.suptitle('Heart Disease EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# TODO 3: Correlation Heatmap
plt.figure(figsize=(12, 8))
corr = df_heart.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Heart Disease — Correlation Heatmap', fontsize=12)
plt.tight_layout()
plt.show()

# Target se correlation dhundo
target_corr = corr['target'].drop('target').sort_values(key=abs, ascending=False)
print("\n📊 Features ranked by correlation with target:")
print(target_corr.to_string())

---
## ✅ Part B — Model Building + Hyperparameter Tuning

In [ ]:
# TODO 4: 3 alag-alag depths pe tree compare karo
results = []
for depth in [3, 5, 7, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_h, y_train_h)
    cv = cross_val_score(dt, X_h, y_h, cv=5)
    results.append({
        'max_depth': str(depth),
        'Tree Depth': dt.get_depth(),
        'Train Acc': round(dt.score(X_train_h, y_train_h), 4),
        'Test Acc':  round(dt.score(X_test_h, y_test_h), 4),
        'CV Mean':   round(cv.mean(), 4),
        'CV Std':    round(cv.std(), 4)
    })

print(pd.DataFrame(results).to_string(index=False))
print("\n📌 depth=None → deepest tree → Train=1.0 → Overfitting!")

In [ ]:
# TODO 5: GridSearchCV
param_grid = {
    'max_depth':         [3, 5, 7, None],
    'min_samples_leaf':  [1, 5, 10],
    'criterion':         ['gini', 'entropy']
}

gs_heart = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
gs_heart.fit(X_train_h, y_train_h)

print(f"Best Params: {gs_heart.best_params_}")
print(f"Best CV Score: {gs_heart.best_score_:.4f}")

best_heart = gs_heart.best_estimator_
y_pred_heart = best_heart.predict(X_test_h)
print(f"Test Accuracy: {accuracy_score(y_test_h, y_pred_heart):.4f}")
print(f"Depth: {best_heart.get_depth()} | Leaves: {best_heart.get_n_leaves()}")

print("\n📋 Classification Report:")
print(classification_report(y_test_h, y_pred_heart, target_names=['No Disease','Disease']))

---
## ✅ Part C — Feature Importance + Tree Visualization

In [ ]:
# TODO 6: Feature Importance
importances_h = pd.Series(best_heart.feature_importances_,
                           index=X_h.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
colors = ['#e74c3c' if v > 0.1 else ('#f39c12' if v > 0.03 else '#bdc3c7') for v in importances_h]
importances_h.plot(kind='barh', color=colors, edgecolor='black', alpha=0.85)
plt.xlabel('Importance (Gini-based)')
plt.title('Feature Importance — Heart Disease Decision Tree')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Top 5 Important Features:")
print(importances_h.sort_values(ascending=False).head(5).to_string())

In [ ]:
# TODO 7: Tree Visualization
plt.figure(figsize=(20, 8))
plot_tree(best_heart,
          feature_names=list(X_h.columns),
          class_names=['No Disease', 'Disease'],
          filled=True,
          rounded=True,
          max_depth=3,
          fontsize=8)
plt.title('Heart Disease Decision Tree (Top 3 Levels)', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📋 Text Rule Extraction:")
print(export_text(best_heart, feature_names=list(X_h.columns), max_depth=3))

---
## ✅ Part D — Interpretability

In [ ]:
# TODO 8: Ek specific patient explain karo
patient_idx = 0
patient = X_test_h.iloc[patient_idx]
actual  = y_test_h.iloc[patient_idx]
pred    = best_heart.predict([patient])[0]
prob    = best_heart.predict_proba([patient])[0]

print("=" * 55)
print("🏥 Patient Profile:")
print("-" * 55)
print(f"  age      : {patient['age']} years")
print(f"  sex      : {'Male' if patient['sex']==1 else 'Female'}")
print(f"  trestbps : {patient['trestbps']} mmHg")
print(f"  chol     : {patient['chol']} mg/dl")
print(f"  thalach  : {patient['thalach']} bpm")
print(f"  ca       : {patient['ca']} major vessels")
print(f"  exang    : {'Yes' if patient['exang']==1 else 'No'}")
print("=" * 55)
print(f"  Actual:    {'⚠️ Has Disease' if actual==1 else '✅ No Disease'}")
print(f"  Predicted: {'⚠️ Has Disease' if pred==1 else '✅ No Disease'}")
print(f"  P(No Disease)={prob[0]:.3f} | P(Disease)={prob[1]:.3f}")
correct = '✅ Correct!' if actual==pred else '❌ Wrong!'
print(f"  Result: {correct}")
print("=" * 55)

In [ ]:
# TODO 9: Galat predictions analyze karo
X_test_reset = X_test_h.reset_index(drop=True)
y_test_reset = y_test_h.reset_index(drop=True)
y_pred_all   = best_heart.predict(X_test_h)

err_df = X_test_reset.copy()
err_df['Actual']    = y_test_reset.values
err_df['Predicted'] = y_pred_all
err_df['Correct']   = (err_df['Actual'] == err_df['Predicted'])

wrong_df = err_df[~err_df['Correct']]
right_df = err_df[err_df['Correct']]

print(f"Total Test: {len(err_df)} | Correct: {len(right_df)} | Wrong: {len(wrong_df)}")
print(f"Accuracy: {len(right_df)/len(err_df):.2%}\n")

# Wrong predictions ka profile
print("=== Galat Predictions Ka Average Profile ===")
compare = pd.DataFrame({
    'Correct Predictions': right_df[['age','trestbps','chol','thalach','ca']].mean(),
    'Wrong Predictions':   wrong_df[['age','trestbps','chol','thalach','ca']].mean()
}).round(2)
print(compare)

# False Negatives (Disease missed)
fn = wrong_df[wrong_df['Actual']==1]
fp = wrong_df[wrong_df['Actual']==0]
print(f"\nFalse Negatives (Disease missed):  {len(fn)} — DANGEROUS for medical!")
print(f"False Positives (Healthy flagged): {len(fp)} — Unnecessary tests")

---
# 📒 Notebook 02 Solutions: Iris Capstone

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)
print(f"Iris: {X_iris.shape} | Train: {X_tr.shape[0]} | Test: {X_te.shape[0]}")
print(f"Classes: {list(iris.target_names)}")

---
## ✅ Part A — EDA

In [ ]:
# TODO 1: Pairplot — kaunse features classes separate karte hain
iris_df = X_iris.copy()
iris_df['species'] = pd.Categorical.from_codes(y_iris, iris.target_names)

sns.pairplot(iris_df, hue='species',
             palette={'setosa': '#e74c3c', 'versicolor': '#3498db', 'virginica': '#2ecc71'},
             plot_kws={'alpha': 0.7}, diag_kind='hist')
plt.suptitle('Iris Pairplot — Class Separation by Features', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print("\n📌 Key Observations:")
print("  - petal length aur petal width sabse clearly classes separate karte hain")
print("  - setosa bilkul alag hai (bottom-left cluster)")
print("  - versicolor aur virginica thoda overlap karte hain petal mein bhi")

---
## ✅ Part B — Depth Analysis

In [ ]:
# TODO 2: Depth 1-10 accuracy plot
depths = range(1, 11)
train_acc_iris, test_acc_iris, cv_acc_iris = [], [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    train_acc_iris.append(dt.score(X_tr, y_tr))
    test_acc_iris.append(dt.score(X_te, y_te))
    cv_acc_iris.append(cross_val_score(dt, X_iris, y_iris, cv=5).mean())

best_d_iris = depths[np.argmax(cv_acc_iris)]

plt.figure(figsize=(9, 4))
plt.plot(depths, train_acc_iris, 'b-o', ms=5, label='Train')
plt.plot(depths, test_acc_iris,  'r-s', ms=5, label='Test')
plt.plot(depths, cv_acc_iris,    'g-^', ms=5, label='CV Mean')
plt.axvline(best_d_iris, color='purple', linestyle='--', lw=2,
            label=f'Best depth={best_d_iris}')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Iris: Depth vs Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best depth: {best_d_iris}")
print(f"At best depth: Train={train_acc_iris[best_d_iris-1]:.3f} | "
      f"Test={test_acc_iris[best_d_iris-1]:.3f} | CV={cv_acc_iris[best_d_iris-1]:.3f}")
print("\n📌 Iris simple dataset hai — depth=3-4 pe hi near-perfect accuracy milti hai!")

---
## ✅ Part C — GridSearchCV

In [ ]:
# TODO 3: GridSearchCV
param_grid_iris = {
    'max_depth':        [2, 3, 5, None],
    'min_samples_leaf': [1, 2, 5],
    'criterion':        ['gini', 'entropy']
}

gs_iris = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_iris,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
gs_iris.fit(X_tr, y_tr)

print(f"Best Params: {gs_iris.best_params_}")
print(f"Best CV Score: {gs_iris.best_score_:.4f}")

best_iris = gs_iris.best_estimator_
y_pred_iris = best_iris.predict(X_te)
print(f"Test Accuracy: {accuracy_score(y_te, y_pred_iris):.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_te, y_pred_iris, target_names=iris.target_names))

---
## ✅ Part D — Visualization + Interpretation

In [ ]:
# TODO 4: Tree Visualization
plt.figure(figsize=(18, 7))
plot_tree(best_iris,
          feature_names=list(iris.feature_names),
          class_names=list(iris.target_names),
          filled=True,
          rounded=True,
          fontsize=10,
          impurity=True)
plt.title(f'Iris Decision Tree (Best: {gs_iris.best_params_})', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📋 Rules:")
print(export_text(best_iris, feature_names=list(iris.feature_names)))

In [ ]:
# TODO 5: Feature Importance
imp_iris = pd.Series(best_iris.feature_importances_,
                     index=iris.feature_names).sort_values(ascending=True)

plt.figure(figsize=(7, 4))
colors = ['#e74c3c' if v > 0.4 else ('#f39c12' if v > 0.1 else '#bdc3c7') for v in imp_iris]
imp_iris.plot(kind='barh', color=colors, edgecolor='black', alpha=0.85)
for i, (name, val) in enumerate(imp_iris.items()):
    plt.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=9)
plt.xlabel('Importance')
plt.title('Iris Feature Importance')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Feature Importances:")
print(imp_iris.sort_values(ascending=False).to_string())

In [ ]:
# TODO 6: EDA ke comparison ke saath insight
print("=" * 60)
print("📌 EDA vs Feature Importance Comparison")
print("=" * 60)
print()
print("EDA se humne dekha tha:")
print("  → petal length aur petal width sabse clearly classes separate karte hain")
print("  → sepal features mein zyada overlap tha")
print()
print("Feature Importance confirm karta hai:")
top_feat = imp_iris.sort_values(ascending=False).index[0]
print(f"  → '{top_feat}' most important feature hai!")
print("  → sepal features ka importance bahut low hai")
print()
print("Conclusion: EDA aur model dono agree karte hain! 🎯")
print("  Yeh confirm karta hai ki feature importance EDA ka hi quantitative version hai.")

# Decision Boundary Visualization (2 features pe)
from sklearn.inspection import DecisionBoundaryDisplay

# Sirf top 2 features use karo visualization ke liye
feature_idx = [2, 3]  # petal length, petal width
X_2d = X_iris.iloc[:, feature_idx]

dt_2d = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_2d.fit(X_2d, y_iris)

plt.figure(figsize=(8, 5))
disp = DecisionBoundaryDisplay.from_estimator(
    dt_2d, X_2d, response_method='predict',
    alpha=0.4, cmap='RdYlGn'
)
scatter = plt.scatter(X_2d.iloc[:, 0], X_2d.iloc[:, 1],
                      c=y_iris, cmap='RdYlGn', edgecolors='black', s=40, alpha=0.8)
plt.xlabel(iris.feature_names[feature_idx[0]])
plt.ylabel(iris.feature_names[feature_idx[1]])
plt.title('Decision Boundary: Petal Length vs Petal Width')
handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8)
           for c in ['#d73027','#fee08b','#1a9850']]
plt.legend(handles, iris.target_names, title='Species')
plt.tight_layout()
plt.show()

---
## 🎯 Final Summary — Dono Notebooks Ka Ek Saath

| Concept | Heart Disease | Iris |
|---------|-------------|------|
| Dataset size | 303 (synthetic) | 150 |
| Task | Binary classification | Multi-class (3 classes) |
| Best depth | GridSearch se mila | Depth=3-4 kafi |
| Top feature | age/ca/trestbps | petal length |
| Accuracy | ~70-80% | ~95%+ |
| Overfitting | Depth>7 pe | Depth>5 pe |

### Key Learnings Recap

1. **Gini = 0** → Perfect pure node → Decision clear hai
2. **max_depth** = single most important parameter — always tune karo
3. **GridSearchCV** = Multiple hyperparameters ek saath tune karne ka best way
4. **Feature Importance** = EDA ka quantitative confirmation
5. **plot_tree** = Model ko explain karne ka best tool clients ke liye
6. **False Negatives** in medical domain = More dangerous than False Positives

### Aage Kya Seekhein?
```
Decision Tree  →  Random Forest  →  Gradient Boosting  →  XGBoost
   (1 tree)        (100+ trees)      (sequential trees)   (optimized)
   Baseline         Better Acc         Even Better        Production
```